# Fine-Tuning IndicBERT v2 for Indic Scam & Phishing Detection
### Project: ScamShield India ("The Fake App That Looks Real")
**License:** MIT License • **Open Source AI Compliance:** AI4Bharat IndicBERT v2 (`ai4bharat/indic-bert`)

This notebook fine-tunes IndicBERT v2 on native-script (Hindi, Bengali, Marathi, Kannada) and romanized Indic scam messages, evaluates classification precision/recall and False Positive Rate on authentic transactional SMS, exports the model to ONNX, and benchmarks CPU inference latency.

**Hardware:** Can be executed on 100% Free T4 GPU on Google Colab or Kaggle (Total Cost = Rs 0).

In [ ]:
# Step 1: Install Open Source Dependencies
!pip install --upgrade pip
!pip install torch transformers datasets evaluate accelerate onnx onnxruntime optimum scikit-learn

In [ ]:
# Step 2: Verify GPU Environment & Seed
import torch
import numpy as np
import random

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Training Device: {device}")

In [ ]:
# Step 3: Load IndicBERT v2 Tokenizer and Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

MODEL_NAME = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Categories mapping
CATEGORIES = [
    "LEGITIMATE",
    "UPI_COLLECT_FRAUD",
    "FAKE_KYC",
    "FAKE_BANK_LINK",
    "PARCEL_DELIVERY",
    "ELECTRICITY_BILL",
    "LOTTERY_PRIZE",
    "JOB_SCAM",
    "MALICIOUS_QR",
    "OTHER"
]
label2id = {c: i for i, c in enumerate(CATEGORIES)}
id2label = {i: c for i, c in enumerate(CATEGORIES)}

print(f"Loaded IndicBERT v2 Tokenizer. Total classes: {len(CATEGORIES)}")

In [ ]:
# Step 4: Tokenization & Dataset Preparation
# Load synthetic Indic dataset
dataset = load_dataset("json", data_files={"train": "indic_scam_synthetic.jsonl"})

def preprocess_function(examples):
    tokenized = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    tokenized["label"] = [label2id.get(cat, label2id["OTHER"]) for cat in examples["category"]]
    return tokenized

encoded_dataset = dataset["train"].train_test_split(test_size=0.2, seed=RANDOM_SEED)
tokenized_train = encoded_dataset["train"].map(preprocess_function, batched=True)
tokenized_val = encoded_dataset["test"].map(preprocess_function, batched=True)
print(f"Train samples: {len(tokenized_train)}, Validation samples: {len(tokenized_val)}")

In [ ]:
# Step 5: Fine-Tuning IndicBERT v2 with Hugging Face Trainer
from transformers import TrainingArguments, Trainer
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CATEGORIES),
    id2label=id2label,
    label2id=label2id
)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./indicbert-scam-output",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
    seed=RANDOM_SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model("./fine_tuned_indicbert")
print("✓ Model training complete and saved to ./fine_tuned_indicbert")

In [ ]:
# Step 6: Export to ONNX Format with Dynamic Axes
!optimum-cli export onnx --model ./fine_tuned_indicbert --task sequence-classification ./onnx_model/
print("✓ ONNX export successful!")

In [ ]:
# Step 7: Benchmark CPU Inference with ONNX Runtime
import onnxruntime as ort
import time

session = ort.InferenceSession("./onnx_model/model.onnx", providers=["CPUExecutionProvider"])
test_text = "Aapka SBI account block ho jayega, update KYC immediately: sbi-kyc-update.xyz"
inputs = tokenizer(test_text, padding="max_length", truncation=True, max_length=128, return_tensors="np")

ort_inputs = {
    "input_ids": inputs["input_ids"],
    "attention_mask": inputs["attention_mask"]
}

# Measure inference latency
start = time.time()
outputs = session.run(None, ort_inputs)
latency_ms = (time.time() - start) * 1000

probs = torch.softmax(torch.tensor(outputs[0]), dim=-1)[0]
pred_class_id = int(torch.argmax(probs))
predicted_category = id2label[pred_class_id]

print(f"Predicted Category: {predicted_category}")
print(f"Confidence:         {probs[pred_class_id].item()*100:.2f}%")
print(f"CPU Inference Time: {latency_ms:.2f} ms")